# 8.4 File Handling — Reading Text Over the Network

**Prerequisites:** 8.1 File Handling — Text, 2.1 Strings (str vs bytes)  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Reading a remote file with `urllib.request`
- 🔴 Why the network gives you **`bytes`**, and where to decode
- Timeouts, `URLError` and `HTTPError` — none of which are optional
- Streaming line by line vs downloading everything
- Writing a remote file to disk
- Why real projects use `requests` instead

---

## Reading a file that lives on a server

Everything in **8.1** applies to local files. A file reached over HTTP is *almost* the same
— `urlopen()` returns an object you can use with `with`, and iterate line by line.

Two things are different, and both catch people out:

1. **You get `bytes`, not `str`.** HTTP transfers raw octets. Nothing has decided the
   encoding for you.
2. **It can fail in ways a local file cannot** — no network, DNS failure, a 404, a 500, or a
   server that accepts the connection and then never responds.

> ### ⚠️ These examples need a working internet connection
> The cells below fetch a small public text file. Each one falls back to a bundled local
> copy if the network is unavailable, so the notebook still runs offline — but the network
> path is the one being taught.

In [ ]:
from urllib.request import urlopen
from urllib.error import URLError, HTTPError

# 🔴 The original notebook used http:// with no timeout and no error handling.
#    Use HTTPS, and ALWAYS pass a timeout - without one, a hung server blocks
#    your program indefinitely.
URL = "https://sixty-north.com/c/t.txt"
TIMEOUT = 10

FALLBACK = (
    b"It was the best of times\n"
    b"it was the worst of times\n"
    b"it was the age of wisdom\n"
    b"it was the age of foolishness\n"
)


def fetch(url: str = URL, timeout: int = TIMEOUT) -> bytes:
    """Fetch a URL, falling back to a local copy if the network is unavailable."""
    try:
        with urlopen(url, timeout=timeout) as response:
            print(f"  HTTP {response.status} | {response.headers.get('Content-Type')}")
            return response.read()
    except HTTPError as exc:            # the server answered, with an error status
        print(f"  HTTP error {exc.code}: {exc.reason} - using the local copy")
    except URLError as exc:             # never reached the server at all
        print(f"  network error: {exc.reason} - using the local copy")
    except TimeoutError:
        print(f"  timed out after {timeout}s - using the local copy")
    return FALLBACK


raw = fetch()
print("\ntype     :", type(raw).__name__)
print("length   :", len(raw), "bytes")
print("first 40 :", raw[:40])

### 🔴 Why you get `bytes`, and where to decode

`response.read()` returns **`bytes`** — the raw octets that came off the wire. To treat them
as text you must **decode** them, which means knowing the encoding (**2.1**).

```python
raw = response.read()          # bytes
text = raw.decode("utf-8")     # str
```

The original notebook did this per line, inside the loop:

```python
for line in story:
    line_words = line.decode('utf-8').split()
```

That works, but it decodes in the wrong place. The **Unicode sandwich** rule from **2.1**
says: *decode at the boundary, work in `str`, encode on the way out.* Decode once, then stop
thinking about bytes.

### Where does the encoding come from?

The server usually tells you, in the `Content-Type` header:

```
Content-Type: text/plain; charset=utf-8
```

`response.headers.get_content_charset()` reads it for you. If the header is missing, you
have to choose — and **UTF-8 is the right default** for anything modern.

In [ ]:
from urllib.request import urlopen
from urllib.error import URLError, HTTPError

# ---- Let the server tell you the encoding, with utf-8 as the fallback ----
def fetch_text(url: str = URL, timeout: int = TIMEOUT) -> str:
    try:
        with urlopen(url, timeout=timeout) as response:
            charset = response.headers.get_content_charset() or "utf-8"
            print(f"  server said charset={response.headers.get_content_charset()!r}"
                  f" -> using {charset!r}")
            return response.read().decode(charset)
    except (URLError, HTTPError, TimeoutError) as exc:
        print(f"  {type(exc).__name__} - using the local copy")
        return FALLBACK.decode("utf-8")


text = fetch_text()
print("\ntype:", type(text).__name__)
print(text)

# ---- Now it is ordinary text, and everything from 2.1 applies ----
words = text.split()
print("words        :", len(words))
print("unique words :", len(set(w.lower().strip('.,') for w in words)))
print("longest word :", max(words, key=len))

from collections import Counter
common = Counter(w.lower().strip(".,") for w in words).most_common(3)
print("most common  :", common)

### Streaming vs downloading everything

`response.read()` pulls the **whole** body into memory. For a 4 KB text file that is fine;
for a 4 GB dataset it is not.

The response object is **iterable**, yielding one line at a time — exactly like a local file
object (**8.1**). Each line is still `bytes`.

| Approach | Memory | Use when |
|---|---|---|
| `response.read()` | Whole body | Small, and you need it all |
| `for line in response:` | One line | Large text, or you can stop early |
| `response.read(n)` in a loop | One chunk | Binary, or precise control (**8.5**) |

In [ ]:
from urllib.request import urlopen
from urllib.error import URLError, HTTPError
import io

# ---- Streaming line by line: constant memory, and you can stop early ----
def stream_lines(url: str = URL, timeout: int = TIMEOUT):
    """Yield decoded lines, from the network or the local fallback."""
    try:
        response = urlopen(url, timeout=timeout)
    except (URLError, HTTPError, TimeoutError) as exc:
        print(f"  {type(exc).__name__} - streaming the local copy instead")
        response = io.BytesIO(FALLBACK)

    with response:
        for raw_line in response:          # each line is BYTES
            yield raw_line.decode("utf-8").rstrip("\n")


print("streaming:")
for n, line in enumerate(stream_lines(), start=1):
    print(f"  {n}: {line}")
    if n == 3:
        print("  ...stopping early - the rest was never transferred")
        break


# ---- Saving a remote file to disk, without loading it all ----
import shutil
from pathlib import Path

target = Path("File2Save/downloaded.txt")

try:
    with urlopen(URL, timeout=TIMEOUT) as response, open(target, "wb") as handle:
        shutil.copyfileobj(response, handle)     # streams in chunks
    print(f"\ndownloaded {target.stat().st_size} bytes to {target.name}")
except (URLError, HTTPError, TimeoutError) as exc:
    target.write_bytes(FALLBACK)
    print(f"\n{type(exc).__name__} - wrote the local copy instead"
          f" ({target.stat().st_size} bytes)")

print("first line:", target.read_text(encoding="utf-8").splitlines()[0])
target.unlink()

---

### Why real projects use `requests`

`urllib.request` is in the standard library, which is its one advantage. For anything beyond
"fetch this URL", it is verbose:

| Task | `urllib.request` | `requests` |
|---|---|---|
| GET and decode | `urlopen(u).read().decode(cs)` | `requests.get(u).text` |
| Parse JSON | `json.loads(...read().decode())` | `requests.get(u).json()` |
| Query parameters | Build the string by hand | `params={"q": "x"}` |
| POST JSON | Build a `Request` with headers | `requests.post(u, json=data)` |
| Headers / auth | Manual `Request` construction | `headers=`, `auth=` |
| Raise on 4xx/5xx | Check `.status` yourself | `response.raise_for_status()` |
| Sessions, retries, connection pooling | Hand-rolled | Built in |

```python
import requests

response = requests.get("https://api.example.com/users", timeout=10)
response.raise_for_status()
users = response.json()
```

`requests` is third-party (`pip install requests`), so it needs a virtual environment
(**7.2**). It gets full treatment in **18 Working with APIs**.

> **Use `urllib` when** you cannot add a dependency — a single-file script, a constrained
> environment, or the standard library is a hard requirement.
> **Use `requests` otherwise.**

---

## Common Mistakes & Pitfalls

1. 🔴 **No timeout.** `urlopen(url)` with no `timeout=` will wait forever on a hung server. Always pass one.
2. 🔴 **No error handling.** Networks fail. Catch `HTTPError` (the server answered with an error) and `URLError` (you never reached it) — `HTTPError` is a subclass, so catch it **first**.
3. **Forgetting to decode.** `response.read()` gives `bytes`; `bytes + str` raises `TypeError` (**2.1**).
4. **Decoding inside the loop** instead of once at the boundary.
5. **Hardcoding `utf-8`** when the server declares something else. Check `headers.get_content_charset()`.
6. **`http://` instead of `https://`.** Plain HTTP is unencrypted and tamperable.
7. **`.read()` on a large response**, exhausting memory. Stream instead.
8. **Assuming a 200 means valid data.** An API can return HTTP 200 with an HTML error page — which is why `json.loads` then fails (**8.3**).
9. **Not closing the response.** Use `with`.

## Best Practices

- **Always** pass `timeout=`.
- **Always** use `https://`.
- Catch `HTTPError` before `URLError`, and handle `TimeoutError`.
- Decode once, at the boundary; work in `str` afterwards.
- Take the encoding from the response headers, defaulting to UTF-8.
- Stream large responses with iteration or `shutil.copyfileobj`.
- Use `with` on the response object.
- Reach for **`requests`** as soon as you need headers, auth, JSON or retries.
- Give network code a **fallback or a retry**, so a transient failure is not fatal.

## Practice Exercises

Try these before moving on.

1. Fetch a URL with `timeout=0.001` and observe which exception you get.
2. Fetch a URL that returns 404 and handle `HTTPError`, printing `.code` and `.reason`.
3. Fetch a URL and count word frequency, decoding exactly once.
4. Stream a remote file and stop after the first line matching a pattern. Confirm the rest was never downloaded.
5. Download a file to disk with `shutil.copyfileobj` and verify its size.
6. Write a `fetch_with_retry(url, attempts=3)` that backs off between attempts.
7. Compare the same GET written with `urllib` and with `requests`. Count the lines.
8. Read the `Content-Type` header of three different URLs and report each charset.